# 05 Distillation (CE + KL)

Cel: wytrenowac student model z polaczeniem CE i KL (soft labels od teacher).


In [113]:
# Krok 1: importy i konfiguracja
# Po co to robimy: przygotowujemy biblioteki i stale, aby uruchomic trening distillation CE + KL.

# Importujemy os do konfiguracji cache i warningow Hugging Face.
import os
# Ustawiamy flage, aby ukryc warning o symlinkach na Windows.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# Ustawiamy lokalny katalog cache HF w repo.
os.environ.setdefault("HF_HOME", "../artifacts/hf_cache")
# Wylaczamy Xet backend, aby nie widziec komunikatu o hf_xet.
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Importujemy random do ustawienia seed dla Pythona.
import random
# Importujemy json do zapisu metryk na dysk.
import json
# Importujemy numpy do pracy z plikami teacher logits.
import numpy as np
# Importujemy PyTorch do treningu modelu.
import torch
# Importujemy funkcje numeryczne PyTorch, miedzy innymi log_softmax i kl_div.
import torch.nn.functional as F
# Importujemy loader danych z dysku.
from datasets import load_from_disk
# Importujemy metryki accuracy i F1 macro.
from sklearn.metrics import accuracy_score, f1_score
# Importujemy clipping gradientow dla stabilizacji treningu.
from torch.nn.utils import clip_grad_norm_
# Importujemy DataLoader do batchowania danych.
from torch.utils.data import DataLoader
# Importujemy pasek postepu dla treningu.
from tqdm.auto import tqdm
# Importujemy model klasyfikacyjny i tokenizer z Transformers.
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Ustalamy seed, aby porownanie baseline vs distillation bylo powtarzalne.
SEED = 42
# Ustawiamy seed dla losowosci Pythona.
random.seed(SEED)
# Ustawiamy seed dla losowosci numpy.
np.random.seed(SEED)
# Ustawiamy seed dla losowosci PyTorch.
torch.manual_seed(SEED)

# Ustalamy nazwe modelu studenta (start pretrained).
STUDENT_MODEL_NAME = "prajjwal1/bert-tiny"
# Ustalamy sciezke warm-start z baseline studenta.
STUDENT_WARM_START_PATH = "../artifacts/student_fp32_baseline"
# Ustalamy, czy wlaczyc warm-start z baseline.
USE_WARM_START = True
# Ustalamy sciezke najlepszego checkpointu podczas treningu.
BEST_CKPT_DIR = "../artifacts/student_fp32_distilled_best"
# Ustalamy rozmiar batcha.
BATCH_SIZE = 8
# Ustalamy liczbe epok distillation.
EPOCHS = 5
# Ustalamy learning rate dla AdamW.
LR = 2e-5
# Ustalamy temperature dla miekkich etykiet teachera.
TEMPERATURE = 1.0
# Ustalamy balans CE vs KL.
ALPHA = 0.8
# Ustalamy maksymalna norme gradientu dla clippingu.
MAX_GRAD_NORM = 1
# Ustalamy urzadzenie CPU zgodnie z zalozeniem projektu.
DEVICE = torch.device("cpu")


In [114]:
# Krok 2: ladowanie danych i teacher logits
# Po co to robimy: ladujemy tokeny studenta i miekkie etykiety teachera, ktore sa niezbedne do distillation.

# Wczytujemy tokenized dane studenta z dysku.
student_ds = load_from_disk("../artifacts/tokenized_student")
# Tworzymy generator PyTorch z seed, aby shuffle train byl powtarzalny.
train_generator = torch.Generator()
# Ustawiamy seed generatora treningowego.
train_generator.manual_seed(SEED)
# Tworzymy loader treningowy z tasowaniem i seedem.
train_loader = DataLoader(student_ds["train"], batch_size=BATCH_SIZE, shuffle=True, generator=train_generator)
# Tworzymy loader walidacyjny bez tasowania.
val_loader = DataLoader(student_ds["validation"], batch_size=BATCH_SIZE, shuffle=False)
# Tworzymy loader testowy bez tasowania.
test_loader = DataLoader(student_ds["test"], batch_size=BATCH_SIZE, shuffle=False)

# Wczytujemy zapisane logits teachera dla splitu train.
teacher_npz = np.load("../artifacts/teacher_logits_train.npz")
# Pobieramy indeksy rekordow, aby dopasowac logits do batcha.
teacher_idx = teacher_npz["idx"]
# Pobieramy tablice logits teachera.
teacher_logits = teacher_npz["logits"]

# Krok 3: mapujemy idx -> logits, aby batch mial szybki dostep
# Po co to robimy: w petli treningowej szybko pobieramy logits teachera dla rekordow z aktualnego batcha.

# Budujemy slownik: klucz to idx rekordu, wartosc to logits teachera.
teacher_map = {int(i): teacher_logits[pos] for pos, i in enumerate(teacher_idx)}


In [115]:
# Krok 4: budujemy student model i optymalizator
# Po co to robimy: przygotowujemy model, optymalizator i CE loss do uczenia studenta.

# Domyslnie startujemy od pretrained studenta.
model_source = STUDENT_MODEL_NAME
# Jesli warm-start jest wlaczony i baseline istnieje, ladujemy baseline jako punkt startowy.
if USE_WARM_START and os.path.exists(os.path.join(STUDENT_WARM_START_PATH, "config.json")):
    model_source = STUDENT_WARM_START_PATH
print("Model source:", model_source)

# Wczytujemy model studenta z wybranego zrodla.
model = AutoModelForSequenceClassification.from_pretrained(model_source, num_labels=3)
# Przenosimy model na CPU.
model.to(DEVICE)
# Tworzymy optymalizator AdamW dla parametrow modelu.
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
# Definiujemy CE jako czesc lossu distillation.
ce_loss_fn = torch.nn.CrossEntropyLoss()


Model source: ../artifacts/student_fp32_baseline


In [116]:
# Krok 5: funkcja distillation loss (CE + KL)
# Po co to robimy: liczymy osobno CE i KL, aby miec lepsza diagnostyke treningu.

# Definiujemy funkcje, ktora zwraca CE, KL i total loss.
def distillation_loss_components(student_logits, labels, teacher_logits_batch, temperature, alpha):
    # Liczymy CE do twardych etykiet.
    ce_loss = ce_loss_fn(student_logits, labels)

    # Dla alpha=1.0 pracujemy w trybie CE-only.
    if alpha >= 1.0:
        # KL w CE-only ustawiamy na zero tensor.
        kd_loss = torch.zeros((), device=student_logits.device, dtype=student_logits.dtype)
        # Total loss to samo CE.
        total_loss = ce_loss
        return ce_loss, kd_loss, total_loss

    # Sprawdzamy, czy logits teachera sa dostepne dla KL.
    if teacher_logits_batch is None:
        raise ValueError("Brak teacher_logits_batch przy alpha < 1.0")

    # Liczymy log-softmax studenta z temperatura.
    log_p_student = F.log_softmax(student_logits / temperature, dim=-1)
    # Liczymy softmax teachera z temperatura.
    p_teacher = F.softmax(teacher_logits_batch / temperature, dim=-1)
    # Liczymy KL divergence i skalujemy przez T^2.
    kd_loss = F.kl_div(log_p_student, p_teacher, reduction="batchmean") * (temperature ** 2)
    # Laczymy CE i KL jednym wspolczynnikiem alpha.
    total_loss = alpha * ce_loss + (1.0 - alpha) * kd_loss
    # Zwracamy CE, KL i total.
    return ce_loss, kd_loss, total_loss


In [117]:
# Krok 6: funkcja ewaluacji
# Po co to robimy: mierzymy jakosc studenta na validation/test bez uczenia i bez gradientow.

# Definiujemy funkcje ewaluacji modelu na podanym loaderze.
def evaluate(model, dataloader):
    # Przelaczamy model w tryb ewaluacji.
    model.eval()
    # Tworzymy liste na predykcje modelu.
    preds = []
    # Tworzymy liste na etykiety referencyjne.
    labels = []
    # Wylaczamy gradienty, bo nie uczymy modelu w ewaluacji.
    with torch.no_grad():
        # Iterujemy po batchach danych.
        for batch in dataloader:
            # Pobieramy input_ids i przenosimy na CPU.
            input_ids = batch["input_ids"].to(DEVICE)
            # Pobieramy attention_mask i przenosimy na CPU.
            attention_mask = batch["attention_mask"].to(DEVICE)
            # Pobieramy etykiety i przenosimy na CPU.
            y = batch["label"].to(DEVICE)
            # Liczymy logits modelu dla batcha.
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            # Zamieniamy logits na klasy i dopisujemy do predykcji.
            preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
            # Dopisujemy prawdziwe etykiety do listy labels.
            labels.extend(y.cpu().tolist())
    # Zwracamy accuracy i F1 macro.
    return {
        # Liczymy accuracy.
        "accuracy": accuracy_score(labels, preds),
        # Liczymy F1 macro.
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


In [118]:
# Krok 7: petla distillation
# Po co to robimy: uczymy studenta i zapisujemy najlepszy checkpoint po val f1.

# Ustawiamy najlepszy wynik walidacyjny na bardzo niska wartosc startowa.
best_val_f1 = -1.0
# Tworzymy liste historii metryk po epokach.
epoch_history = []

# Iterujemy po epokach distillation.
for epoch in range(EPOCHS):
    # Przelaczamy model w tryb treningu.
    model.train()
    # Tworzymy pasek postepu dla batchy treningowych.
    pbar = tqdm(train_loader, desc=f"Distill epoch {epoch + 1}/{EPOCHS}")
    # Tworzymy zmienne do srednich CE/KL/total po epoce.
    epoch_ce = 0.0
    epoch_kd = 0.0
    epoch_total = 0.0
    epoch_steps = 0

    # Iterujemy po batchach treningowych.
    for batch in pbar:
        # Pobieramy input_ids i przenosimy na CPU.
        input_ids = batch["input_ids"].to(DEVICE)
        # Pobieramy attention_mask i przenosimy na CPU.
        attention_mask = batch["attention_mask"].to(DEVICE)
        # Pobieramy twarde etykiety i przenosimy na CPU.
        labels = batch["label"].to(DEVICE)

        # Przy alpha<1.0 przygotowujemy logits teachera dla batcha.
        if ALPHA < 1.0:
            # Pobieramy indeksy rekordow z batcha.
            idx = batch["idx"].cpu().tolist()
            # Budujemy tensor logits teachera dla aktualnego batcha.
            teacher_logits_batch = torch.tensor(
                [teacher_map[int(i)] for i in idx],
                dtype=torch.float32,
                device=DEVICE,
            )
        else:
            # W CE-only nie potrzebujemy logits teachera.
            teacher_logits_batch = None

        # Liczymy logits studenta dla aktualnego batcha.
        student_logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        # Liczymy CE, KL i total loss.
        ce_loss, kd_loss, loss = distillation_loss_components(
            student_logits=student_logits,
            labels=labels,
            teacher_logits_batch=teacher_logits_batch,
            temperature=TEMPERATURE,
            alpha=ALPHA,
        )

        # Backward pass liczy gradienty parametrow studenta.
        loss.backward()
        # Clipping ogranicza zbyt duze gradienty i stabilizuje uczenie.
        clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        # Optimizer step aktualizuje wagi studenta.
        optimizer.step()
        # Zerujemy gradienty przed kolejnym krokiem.
        optimizer.zero_grad(set_to_none=True)

        # Aktualizujemy sumy CE/KL/total dla statystyk epoki.
        epoch_ce += float(ce_loss.detach().cpu())
        epoch_kd += float(kd_loss.detach().cpu())
        epoch_total += float(loss.detach().cpu())
        epoch_steps += 1

        # Pokazujemy biezace wartosci na pasku postepu.
        pbar.set_postfix({
            "loss": round(float(loss.detach().cpu()), 4),
            "ce": round(float(ce_loss.detach().cpu()), 4),
            "kd": round(float(kd_loss.detach().cpu()), 4),
        })

    # Liczymy metryki walidacyjne po epoce.
    val_metrics_epoch = evaluate(model, val_loader)
    # Liczymy srednie CE/KL/total po epoce.
    mean_ce = epoch_ce / max(1, epoch_steps)
    mean_kd = epoch_kd / max(1, epoch_steps)
    mean_total = epoch_total / max(1, epoch_steps)

    # Zapisujemy historie epoki do pozniejszej analizy.
    epoch_history.append({
        "epoch": epoch + 1,
        "val_accuracy": val_metrics_epoch["accuracy"],
        "val_f1_macro": val_metrics_epoch["f1_macro"],
        "mean_ce_loss": mean_ce,
        "mean_kd_loss": mean_kd,
        "mean_total_loss": mean_total,
    })

    # Wypisujemy podsumowanie epoki.
    print(f"Epoch {epoch+1}: val_f1={val_metrics_epoch["f1_macro"]:.4f}, CE={mean_ce:.4f}, KL={mean_kd:.4f}, total={mean_total:.4f}")

    # Jesli val f1 sie poprawilo, zapisujemy najlepszy checkpoint.
    if val_metrics_epoch["f1_macro"] > best_val_f1:
        # Aktualizujemy najlepsze val f1.
        best_val_f1 = val_metrics_epoch["f1_macro"]
        # Tworzymy katalog best checkpointu, jesli nie istnieje.
        os.makedirs(BEST_CKPT_DIR, exist_ok=True)
        # Zapisujemy najlepszy model po val f1.
        model.save_pretrained(BEST_CKPT_DIR)
        # Zapisujemy tokenizer do tego samego katalogu.
        AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME).save_pretrained(BEST_CKPT_DIR)
        print("Nowy najlepszy checkpoint zapisany do:", BEST_CKPT_DIR)

# Po treningu wczytujemy najlepszy checkpoint, jesli istnieje.
if os.path.exists(os.path.join(BEST_CKPT_DIR, "config.json")):
    model = AutoModelForSequenceClassification.from_pretrained(BEST_CKPT_DIR).to(DEVICE)
    print("Wczytano najlepszy checkpoint z:", BEST_CKPT_DIR)

# Liczymy finalne metryki na validation i test.
val_metrics = evaluate(model, val_loader)
test_metrics = evaluate(model, test_loader)
print("Validation:", val_metrics)
print("Test:", test_metrics)


Distill epoch 1/5:   0%|          | 0/485 [00:00<?, ?it/s]

Epoch 1: val_f1=0.6636, CE=0.6052, KL=0.5413, total=0.5924
Nowy najlepszy checkpoint zapisany do: ../artifacts/student_fp32_distilled_best


Distill epoch 2/5:   0%|          | 0/485 [00:00<?, ?it/s]

Epoch 2: val_f1=0.7163, CE=0.5578, KL=0.4937, total=0.5450
Nowy najlepszy checkpoint zapisany do: ../artifacts/student_fp32_distilled_best


Distill epoch 3/5:   0%|          | 0/485 [00:00<?, ?it/s]

Epoch 3: val_f1=0.7294, CE=0.4926, KL=0.4313, total=0.4803
Nowy najlepszy checkpoint zapisany do: ../artifacts/student_fp32_distilled_best


Distill epoch 4/5:   0%|          | 0/485 [00:00<?, ?it/s]

Epoch 4: val_f1=0.7548, CE=0.4254, KL=0.3753, total=0.4154
Nowy najlepszy checkpoint zapisany do: ../artifacts/student_fp32_distilled_best


Distill epoch 5/5:   0%|          | 0/485 [00:00<?, ?it/s]

Epoch 5: val_f1=0.7336, CE=0.3915, KL=0.3394, total=0.3811
Wczytano najlepszy checkpoint z: ../artifacts/student_fp32_distilled_best
Validation: {'accuracy': 0.8061855670103093, 'f1_macro': 0.7548402697777409}
Test: {'accuracy': 0.7876288659793814, 'f1_macro': 0.7414279650541772}


In [7]:
# Krok 8: zapis distilled modelu i metryk
# Po co to robimy: zapisujemy model po distillation, metryki i historie epok do analizy.

# Ustalamy katalog zapisu finalnego modelu distilled.
save_dir = "../artifacts/student_fp32_distilled"
# Wczytujemy tokenizer studenta do zapisu obok modelu.
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME)
# Zapisujemy model distilled na dysk.
model.save_pretrained(save_dir)
# Zapisujemy tokenizer na dysk.
tokenizer.save_pretrained(save_dir)

# Otwieramy plik JSON do zapisu metryk distilled.
with open("../outputs/metrics_student_distilled.json", "w", encoding="utf-8") as f:
    # Zapisujemy metryki validation i test w czytelnym formacie.
    json.dump({"validation": val_metrics, "test": test_metrics}, f, indent=2)

# Zapisujemy historie epok (val_f1, srednie CE/KL/total).
with open("../outputs/metrics_student_distilled_history.json", "w", encoding="utf-8") as f:
    json.dump(epoch_history, f, indent=2)

# Zapisujemy informacje o najlepszym checkpointcie.
with open("../outputs/student_distilled_best_ckpt.txt", "w", encoding="utf-8") as f:
    f.write(BEST_CKPT_DIR + "\n")

# Potwierdzamy, ze zapis modelu i metryk sie udal.
print("Zapisano distilled model i metryki")


Zapisano distilled model i metryki
